# India AQI Transformer — 01: Data Exploration & Dataset
**Person 1 | Owns: `01_data_exploration.ipynb` · `dataset.py`**

One line summary: gets the raw CSV files from Kaggle into clean, model-ready tensors.

| Stage | What happens here |
|---|---|
| Upload | Zip the Dataset folder → upload → extract to Google Drive |
| Load | Read all ~453 per-station CSVs, rename columns, concatenate |
| Explore | Per-station missing data audit, PM2.5 distributions, seasonality |
| Clean | Drop bad stations, per-station interpolation, column pruning |
| Normalise | StandardScaler per-pollutant, fit on train only |
| Package | `AQIDataset` (per-station sliding window) + `build_dataloaders` |
| Deliver | `dataset.py` written to Google Drive — share the folder with teammates |

> **Target variables: all 7 pollutants simultaneously** — PM2.5, PM10, NO2, NH3, SO2, CO, O3 (surviving columns after sparse-column pruning). The model performs multi-output forecasting: every forward pass predicts the next 48 hours for all pollutants at once.

> **Window sizes (per proposal):** 72-hour input look-back (`seq_len = 72`) → 48-hour forecast horizon (`pred_len = 48`).

---
## 0 · Environment Setup
Re-run every cell in this section at the start of every Colab session.

In [ ]:
# 0.1 — Verify GPU
# Runtime → Change runtime type → T4 GPU → Save
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: No GPU detected. Enable T4 GPU in Runtime settings.')

In [ ]:
# 0.2 — Install dependencies
!pip install -q torch torchvision pandas numpy scikit-learn matplotlib seaborn joblib

In [ ]:
# 0.3 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os

DRIVE_ROOT = '/content/drive/MyDrive/AQI_Project'
DATA_DIR   = f'{DRIVE_ROOT}/data'        # station CSVs extracted here
CKPT_DIR   = f'{DRIVE_ROOT}/checkpoints'

os.makedirs(DATA_DIR,  exist_ok=True)
os.makedirs(CKPT_DIR,  exist_ok=True)
print('Drive mounted. Data dir:', DATA_DIR)

In [ ]:
# 0.4 — Upload the Dataset ZIP and extract it to Drive
# Skipped automatically if the data is already on Drive.

import zipfile, shutil
from google.colab import files

# Check if data already uploaded (at least one station CSV present)
already_uploaded = any(f.endswith('.csv') and f != 'stations_info.csv'
                       for f in os.listdir(DATA_DIR))

if not already_uploaded:
    print('Select the zip file from your computer...')
    uploaded = files.upload()

    zip_name = list(uploaded.keys())[0]
    print(f'Extracting {zip_name}...')

    with zipfile.ZipFile(zip_name, 'r') as z:
        for member in z.namelist():
            if member.endswith('.csv'):
                # flatten any folder structure — put all CSVs directly in DATA_DIR
                filename = os.path.basename(member)
                with z.open(member) as src, open(os.path.join(DATA_DIR, filename), 'wb') as dst:
                    shutil.copyfileobj(src, dst)

    os.remove(zip_name)
    n = sum(1 for f in os.listdir(DATA_DIR) if f.endswith('.csv'))
    print(f'Done. {n} CSV files saved to {DATA_DIR}')
else:
    n = sum(1 for f in os.listdir(DATA_DIR) if f.endswith('.csv'))
    print(f'Data already on Drive ({n} CSV files) — skipping upload.')

---
## 1 · Load & Merge All Station Files

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted')
%matplotlib inline

# ── Column name mapping (raw CSV → clean name) ───────────────────────────────
COL_RENAME = {
    'From Date':           'date',
    'PM2.5 (ug/m3)':      'PM2.5',
    'PM10 (ug/m3)':       'PM10',
    'NO (ug/m3)':         'NO',
    'NO2 (ug/m3)':        'NO2',
    'NOx (ppb)':          'NOx',
    'NH3 (ug/m3)':        'NH3',
    'SO2 (ug/m3)':        'SO2',
    'CO (mg/m3)':         'CO',
    'Ozone (ug/m3)':      'O3',
    'Benzene (ug/m3)':    'Benzene',
    'Toluene (ug/m3)':    'Toluene',
}

STATION_COL    = 'StationId'
DATE_COL       = 'date'
TARGET_COL     = 'PM2.5'          # kept for EDA plots only
POLLUTANT_COLS = ['PM2.5', 'PM10', 'NO', 'NO2', 'NOx',
                  'NH3', 'SO2', 'CO', 'O3', 'Benzene', 'Toluene']
# TARGET_COLS (all 7 surviving pollutants) is derived in Cell 3.2 after sparse-column pruning
# ─────────────────────────────────────────────────────────────────────────────

station_files = sorted([
    f for f in os.listdir(DATA_DIR)
    if f.endswith('.csv') and f != 'stations_info.csv'
])
print(f'Station files found: {len(station_files)}')

frames = []
for fname in station_files:
    station_id = fname.replace('.csv', '')
    sdf = pd.read_csv(os.path.join(DATA_DIR, fname), parse_dates=['From Date'])
    # keep only columns present in this file
    rename = {k: v for k, v in COL_RENAME.items() if k in sdf.columns}
    sdf = sdf[list(rename.keys())].rename(columns=rename)
    sdf[STATION_COL] = station_id
    frames.append(sdf)

df_raw = pd.concat(frames, ignore_index=True)
print(f'Total rows: {len(df_raw):,}')
df_raw.head()

In [ ]:
df_raw.dtypes

---
## 2 · Exploratory Data Analysis

In [ ]:
# 2.1 — Summary statistics
df_raw[POLLUTANT_COLS].describe()

In [ ]:
# 2.2 — Rows per station
station_counts = df_raw[STATION_COL].value_counts()
print(f'Total stations : {station_counts.shape[0]}')
print(f'Total rows     : {len(df_raw):,}')
print(f'\nRows per station (top 10):')
print(station_counts.head(10).to_string())

plt.figure(figsize=(14, 3))
station_counts.sort_values().plot(kind='barh', title='Rows per Station')
plt.xlabel('Number of rows')
plt.tight_layout()
plt.show()

In [ ]:
# 2.3 — Global missing value audit
global_missing = df_raw[POLLUTANT_COLS].isnull().mean().sort_values(ascending=False).round(4)
print('Global missing fraction per pollutant:')
print(global_missing.to_string())

global_missing.plot(
    kind='bar', figsize=(12, 4),
    title='Global Missing Fraction by Pollutant',
    ylabel='Fraction missing'
)
plt.axhline(0.5, color='red', linestyle='--', label='50% drop threshold')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# 2.4 — Per-station missing data analysis
# Identify stations too sparse to be useful.
station_missing = (
    df_raw.groupby(STATION_COL)[POLLUTANT_COLS]
    .apply(lambda g: g.isnull().mean().mean())
    .sort_values(ascending=False)
)

STATION_MISSING_THRESHOLD = 0.40

bad_stations  = station_missing[station_missing >  STATION_MISSING_THRESHOLD]
good_stations = station_missing[station_missing <= STATION_MISSING_THRESHOLD]

print(f'Stations to KEEP (≤{STATION_MISSING_THRESHOLD*100:.0f}% missing): {len(good_stations)}')
print(f'Stations to DROP  (>{STATION_MISSING_THRESHOLD*100:.0f}% missing): {len(bad_stations)}')
print(f'\nWorst 15 stations by missing %:')
print((station_missing.head(15) * 100).round(1).to_string())

plt.figure(figsize=(14, 4))
station_missing.sort_values().plot(
    title='Per-Station Mean Missing Fraction', ylabel='Fraction missing'
)
plt.axhline(STATION_MISSING_THRESHOLD, color='red', linestyle='--',
            label=f'Threshold ({STATION_MISSING_THRESHOLD})')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# 2.5 — PM2.5 distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

df_raw[TARGET_COL].dropna().plot(
    ax=axes[0], alpha=0.4, title='PM2.5 Over Time (all stations)', ylabel='PM2.5 (μg/m³)'
)
df_raw[TARGET_COL].dropna().hist(ax=axes[1], bins=80)
axes[1].set_title('PM2.5 Distribution')
axes[1].set_xlabel('PM2.5 (μg/m³)')

plt.tight_layout()
plt.show()
print(df_raw[TARGET_COL].describe())

In [ ]:
# 2.6 — PM2.5 time series for 4 sample stations
sample_stations = good_stations.index[:4].tolist()
fig, axes = plt.subplots(4, 1, figsize=(14, 12), sharex=True)

for ax, station in zip(axes, sample_stations):
    sdf = df_raw[df_raw[STATION_COL] == station].sort_values(DATE_COL)
    ax.plot(sdf[DATE_COL], sdf[TARGET_COL], linewidth=0.6)
    ax.set_title(f'Station: {station}', fontsize=9)
    ax.set_ylabel('PM2.5')

plt.suptitle('PM2.5 Time Series — Sample Stations', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# 2.7 — Pollutant correlation matrix
df_good = df_raw[df_raw[STATION_COL].isin(good_stations.index)]
corr = df_good[POLLUTANT_COLS].corr()

plt.figure(figsize=(12, 10))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5)
plt.title('Pollutant Correlation Matrix (good stations only)')
plt.tight_layout()
plt.show()

In [ ]:
# 2.8 — Monthly seasonality
df_good = df_raw[df_raw[STATION_COL].isin(good_stations.index)].copy()
df_good['month'] = df_good[DATE_COL].dt.month
monthly_avg = df_good.groupby('month')[TARGET_COL].mean()

monthly_avg.plot(
    kind='bar', figsize=(10, 4),
    title='Average PM2.5 by Month (all good stations)',
    ylabel='Mean PM2.5 (μg/m³)', xlabel='Month'
)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

---
## 3 · Cleaning & Preprocessing

In [ ]:
# 3.1 — Keep good stations, sort chronologically within each
df = df_raw[df_raw[STATION_COL].isin(good_stations.index)].copy()
df = df.sort_values([STATION_COL, DATE_COL]).reset_index(drop=True)

print(f'Shape after station filter : {df.shape}')
print(f'Stations remaining         : {df[STATION_COL].nunique()}')

In [ ]:
# 3.2 — Drop pollutant columns with >50% global missing
COL_MISSING_THRESHOLD = 0.50
col_missing = df[POLLUTANT_COLS].isnull().mean()
bad_cols    = col_missing[col_missing > COL_MISSING_THRESHOLD].index.tolist()
print('Dropping sparse columns:', bad_cols)

df = df.drop(columns=bad_cols)

# All surviving pollutant columns are BOTH model inputs AND outputs (multi-target forecasting).
# Typically yields 7 columns: PM2.5, PM10, NO2, NH3, SO2, CO, O3
ALL_COLS     = [c for c in POLLUTANT_COLS if c not in bad_cols]
FEATURE_COLS = ALL_COLS   # encoder input: all pollutants at each past timestep
TARGET_COLS  = ALL_COLS   # decoder output: all pollutants for each future timestep

print(f'Pollutant columns — input & target ({len(ALL_COLS)} total): {ALL_COLS}')

In [ ]:
# 3.3 — Interpolate within each station's time series
#
# IMPORTANT: scoped per station — never fills across station boundaries.
num_cols = ALL_COLS   # interpolate every pollutant column

df[num_cols] = (
    df.groupby(STATION_COL)[num_cols]
    .transform(lambda g: g.interpolate(method='linear', limit_direction='both'))
)
df[num_cols] = (
    df.groupby(STATION_COL)[num_cols]
    .transform(lambda g: g.ffill().bfill())
)

print(f'Missing after interpolation: {df[num_cols].isnull().sum().sum()}')
df = df.dropna(subset=TARGET_COLS).reset_index(drop=True)
print(f'Final shape: {df.shape}')

In [ ]:
# 3.4 — Chronological train / val / test split (global date cutoffs)
#
# Global cutoffs ensure every station's future lands in test — not row-count fractions
# which would give different calendar periods to different stations.
TRAIN_FRAC = 0.70
VAL_FRAC   = 0.15

dates = df[DATE_COL].sort_values()
train_cutoff = dates.quantile(TRAIN_FRAC)
val_cutoff   = dates.quantile(TRAIN_FRAC + VAL_FRAC)

train_df = df[df[DATE_COL] <= train_cutoff].copy()
val_df   = df[(df[DATE_COL] > train_cutoff) & (df[DATE_COL] <= val_cutoff)].copy()
test_df  = df[df[DATE_COL] > val_cutoff].copy()

print(f'Train cutoff : {train_cutoff}')
print(f'Val   cutoff : {val_cutoff}')
print(f'Train rows   : {len(train_df):,}')
print(f'Val   rows   : {len(val_df):,}')
print(f'Test  rows   : {len(test_df):,}')

In [ ]:
# 3.5 — Normalise all pollutants with one shared scaler (fit on train only)
# Single scaler covers every pollutant column (inputs = outputs for multi-target forecasting).
from sklearn.preprocessing import StandardScaler

all_scaler = StandardScaler()

train_df = train_df.copy()
val_df   = val_df.copy()
test_df  = test_df.copy()

train_df[ALL_COLS] = all_scaler.fit_transform(train_df[ALL_COLS])
val_df[ALL_COLS]   = all_scaler.transform(val_df[ALL_COLS])
test_df[ALL_COLS]  = all_scaler.transform(test_df[ALL_COLS])

print('Per-pollutant scaling (train stats):')
for col, mean, std in zip(ALL_COLS, all_scaler.mean_, all_scaler.scale_):
    print(f'  {col:<10} mean={mean:8.3f}  std={std:8.3f}')

---
## 4 · `AQIDataset` — Per-Station Sliding Window

**Why per-station?** With ~450 stations, a naive sliding window on the full concatenated DataFrame would create windows that span two different stations. Every window must stay within one station's timeline.

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader


class AQIDataset(Dataset):
    """
    Per-station sliding-window Dataset for multi-pollutant forecasting.

    Predicts ALL pollutants simultaneously (multi-target, multi-step).

    Args:
        df           : cleaned & normalised DataFrame
        station_col  : column identifying each station
        feature_cols : list of input feature column names  (all pollutants)
        target_cols  : list of output target column names  (all pollutants)
        seq_len      : look-back window in time steps  (72 hrs per proposal)
        pred_len     : forecast horizon in time steps  (48 hrs per proposal)

    Returns per __getitem__:
        x : FloatTensor  (seq_len,  n_features)   — past 72 hrs, all pollutants
        y : FloatTensor  (pred_len, n_targets)     — next 48 hrs, all pollutants
    """

    def __init__(self, df, station_col, feature_cols, target_cols,
                 seq_len=72, pred_len=48):
        self.seq_len  = seq_len
        self.pred_len = pred_len
        self._feat    = {}
        self._tgt     = {}
        self._indices = []

        for station, grp in df.groupby(station_col):
            grp = grp.sort_values(DATE_COL).reset_index(drop=True)
            self._feat[station] = torch.FloatTensor(grp[feature_cols].values)
            self._tgt[station]  = torch.FloatTensor(grp[target_cols].values)
            n = len(grp)
            for i in range(n - seq_len - pred_len + 1):
                self._indices.append((station, i))

        if not self._indices:
            raise ValueError(
                f'No valid windows (seq_len={seq_len}, pred_len={pred_len}). '
                'All stations have fewer rows than seq_len + pred_len.'
            )

    def __len__(self):  return len(self._indices)

    def __getitem__(self, idx):
        station, i = self._indices[idx]
        x = self._feat[station][i : i + self.seq_len]
        y = self._tgt[station][i + self.seq_len : i + self.seq_len + self.pred_len]
        return x, y

    @property
    def n_features(self):  return next(iter(self._feat.values())).shape[1]

    @property
    def n_targets(self):   return next(iter(self._tgt.values())).shape[1]

    @property
    def n_stations(self):  return len(self._feat)


def build_dataloaders(train_df, val_df, test_df, station_col, feature_cols,
                      target_cols, seq_len=72, pred_len=48, batch_size=64, num_workers=2):
    ds_kw = dict(station_col=station_col, feature_cols=feature_cols,
                 target_cols=target_cols, seq_len=seq_len, pred_len=pred_len)
    dl_kw = dict(batch_size=batch_size, num_workers=num_workers, pin_memory=True)
    return (
        DataLoader(AQIDataset(train_df, **ds_kw), shuffle=True,  **dl_kw),
        DataLoader(AQIDataset(val_df,   **ds_kw), shuffle=False, **dl_kw),
        DataLoader(AQIDataset(test_df,  **ds_kw), shuffle=False, **dl_kw),
    )


print('AQIDataset and build_dataloaders defined.')

---
## 5 · Smoke-Test

In [ ]:
SEQ_LEN    = 72   # 72-hour input window  (per proposal)
PRED_LEN   = 48   # 48-hour forecast horizon (per proposal)
BATCH_SIZE = 64

train_loader, val_loader, test_loader = build_dataloaders(
    train_df, val_df, test_df,
    station_col=STATION_COL, feature_cols=FEATURE_COLS,
    target_cols=TARGET_COLS, seq_len=SEQ_LEN, pred_len=PRED_LEN,
    batch_size=BATCH_SIZE,
)

print(f'Training windows   : {len(train_loader.dataset):,}')
print(f'Validation windows : {len(val_loader.dataset):,}')
print(f'Test windows       : {len(test_loader.dataset):,}')
print(f'n_features         : {train_loader.dataset.n_features}')
print(f'n_targets          : {train_loader.dataset.n_targets}')
print(f'n_stations (train) : {train_loader.dataset.n_stations}')

In [ ]:
x_batch, y_batch = next(iter(train_loader))
print('x shape:', x_batch.shape)   # (batch, 72, n_features)
print('y shape:', y_batch.shape)   # (batch, 48, n_targets)  ← all 7 pollutants
print('dtype  :', x_batch.dtype)

In [ ]:
# Plot one sample window vs target for every predicted pollutant
sample_x, sample_y = train_loader.dataset[0]   # x:(72, n_feat)  y:(48, n_tgt)

fig, axes = plt.subplots(len(TARGET_COLS), 1,
                         figsize=(13, 2.5 * len(TARGET_COLS)), sharex=False)

for ax, col in zip(axes, TARGET_COLS):
    col_idx = FEATURE_COLS.index(col)
    ax.plot(range(SEQ_LEN), sample_x[:, col_idx].numpy(),
            label='Input (72 h)', linewidth=0.9)
    ax.plot(range(SEQ_LEN, SEQ_LEN + PRED_LEN), sample_y[:, col_idx].numpy(),
            'o--', color='tomato', markersize=3, label='Target (48 h)')
    ax.axvline(SEQ_LEN - 0.5, color='gray', linestyle=':', alpha=0.5)
    ax.set_title(col, fontsize=9)
    ax.set_ylabel('Norm. val')
    ax.legend(fontsize=7, loc='upper right')

fig.suptitle('Sample Training Window — 72 h Input → 48 h Target (all 7 pollutants, normalised)',
             y=1.01)
plt.tight_layout()
plt.show()

---
## 6 · Save Scaler to Google Drive
Person 4 needs `all_scaler.pkl` to inverse-transform predictions back to real-unit values for all 7 pollutants.

In [ ]:
import joblib

joblib.dump(all_scaler, os.path.join(CKPT_DIR, 'all_scaler.pkl'))
print('Scaler saved to:', CKPT_DIR)
print('Covers columns :', ALL_COLS)

---
## 7 · Write `dataset.py` to Google Drive

After this cell runs, teammates import it with:
```python
import sys
sys.path.insert(0, '/content/drive/MyDrive/AQI_Project')
from dataset import build_dataloaders
train_loader, val_loader, test_loader, scaler, feat_cols = build_dataloaders(DATA_DIR)
# x: (batch, 72, n_features)   y: (batch, 48, n_targets)
# scaler.inverse_transform(...)  → real-unit values for all 7 pollutants
```

In [ ]:
%%writefile /content/drive/MyDrive/AQI_Project/dataset.py
"""
dataset.py — Person 1 deliverable
India AQI Transformer Project

Multi-target forecasting: predicts ALL 7 pollutants simultaneously.
Window sizes: 72-hour input (seq_len) → 48-hour output (pred_len).

Exports:
    AQIDataset        : PyTorch Dataset (per-station sliding window, multi-target)
    build_dataloaders : full pipeline data_dir → (train_loader, val_loader, test_loader, ...)

Usage (Person 3 in train.py):
    import sys
    sys.path.insert(0, '/content/drive/MyDrive/AQI_Project')
    from dataset import build_dataloaders

    DATA_DIR = '/content/drive/MyDrive/AQI_Project/data'
    train_loader, val_loader, test_loader, scaler, feat_cols = build_dataloaders(DATA_DIR)
    # x: (batch, 72, n_features)   y: (batch, 48, n_targets)
    # scaler.inverse_transform(arr_2d)  → real-unit values for all pollutants
"""

import os
import joblib
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler

# ── Column mapping from raw CSV headers to clean names ────────────────────────
COL_RENAME = {
    'From Date':           'date',
    'PM2.5 (ug/m3)':      'PM2.5',
    'PM10 (ug/m3)':       'PM10',
    'NO (ug/m3)':         'NO',
    'NO2 (ug/m3)':        'NO2',
    'NOx (ppb)':          'NOx',
    'NH3 (ug/m3)':        'NH3',
    'SO2 (ug/m3)':        'SO2',
    'CO (mg/m3)':         'CO',
    'Ozone (ug/m3)':      'O3',
    'Benzene (ug/m3)':    'Benzene',
    'Toluene (ug/m3)':    'Toluene',
}

STATION_COL    = 'StationId'
DATE_COL       = 'date'
POLLUTANT_COLS = ['PM2.5', 'PM10', 'NO', 'NO2', 'NOx',
                  'NH3', 'SO2', 'CO', 'O3', 'Benzene', 'Toluene']

STATION_MISSING_THRESHOLD = 0.40
COL_MISSING_THRESHOLD     = 0.50
TRAIN_FRAC = 0.70
VAL_FRAC   = 0.15


# ── Data loading & cleaning ───────────────────────────────────────────────────

def _load_and_clean(data_dir: str):
    """
    Read all per-station CSVs, merge, clean, and return a DataFrame plus
    the list of surviving pollutant column names (used as both input and output).
    """
    station_files = sorted([
        f for f in os.listdir(data_dir)
        if f.endswith('.csv') and f != 'stations_info.csv'
    ])

    frames = []
    for fname in station_files:
        station_id = fname.replace('.csv', '')
        sdf = pd.read_csv(
            os.path.join(data_dir, fname),
            parse_dates=['From Date'],
        )
        rename = {k: v for k, v in COL_RENAME.items() if k in sdf.columns}
        sdf = sdf[list(rename.keys())].rename(columns=rename)
        sdf[STATION_COL] = station_id
        frames.append(sdf)

    df = pd.concat(frames, ignore_index=True)
    df = df.sort_values([STATION_COL, DATE_COL]).reset_index(drop=True)

    # Remove stations with too much missing data
    present = [c for c in POLLUTANT_COLS if c in df.columns]
    station_missing = (
        df.groupby(STATION_COL)[present]
        .apply(lambda g: g.isnull().mean().mean())
    )
    good = station_missing[station_missing <= STATION_MISSING_THRESHOLD].index
    df   = df[df[STATION_COL].isin(good)].copy()

    # Drop sparse pollutant columns (>50% missing globally)
    col_miss  = df[present].isnull().mean()
    bad_cols  = col_miss[col_miss > COL_MISSING_THRESHOLD].index.tolist()
    df        = df.drop(columns=bad_cols)
    # All surviving columns are both model inputs AND outputs (multi-target)
    all_cols  = [c for c in present if c not in bad_cols]

    # Per-station interpolation (never across station boundaries)
    df[all_cols] = (
        df.groupby(STATION_COL)[all_cols]
        .transform(lambda g: g.interpolate(method='linear', limit_direction='both'))
    )
    df[all_cols] = (
        df.groupby(STATION_COL)[all_cols]
        .transform(lambda g: g.ffill().bfill())
    )

    df = df.dropna(subset=all_cols).reset_index(drop=True)
    return df, all_cols


def _split_by_time(df, train_frac, val_frac):
    dates        = df[DATE_COL].sort_values()
    train_cutoff = dates.quantile(train_frac)
    val_cutoff   = dates.quantile(train_frac + val_frac)
    train = df[df[DATE_COL] <= train_cutoff].copy()
    val   = df[(df[DATE_COL] > train_cutoff) & (df[DATE_COL] <= val_cutoff)].copy()
    test  = df[df[DATE_COL] > val_cutoff].copy()
    return train, val, test


def _scale(train, val, test, all_cols):
    """Single StandardScaler covering all pollutant columns (inputs = outputs)."""
    scaler = StandardScaler()
    train, val, test = train.copy(), val.copy(), test.copy()
    train[all_cols] = scaler.fit_transform(train[all_cols])
    val[all_cols]   = scaler.transform(val[all_cols])
    test[all_cols]  = scaler.transform(test[all_cols])
    return train, val, test, scaler


# ── Dataset ───────────────────────────────────────────────────────────────────

class AQIDataset(Dataset):
    """
    Per-station sliding-window Dataset for multi-pollutant forecasting.
    No window ever spans two different monitoring stations.

    Returns per __getitem__:
        x : FloatTensor  (seq_len,  n_features)  — past 72 hrs, all pollutants
        y : FloatTensor  (pred_len, n_targets)   — next 48 hrs, all pollutants
    """

    def __init__(self, df, station_col, feature_cols, target_cols,
                 seq_len=72, pred_len=48):
        self.seq_len  = seq_len
        self.pred_len = pred_len
        self._feat    = {}
        self._tgt     = {}
        self._indices = []

        for station, grp in df.groupby(station_col):
            grp = grp.sort_values(DATE_COL).reset_index(drop=True)
            self._feat[station] = torch.FloatTensor(grp[feature_cols].values)
            self._tgt[station]  = torch.FloatTensor(grp[target_cols].values)
            n = len(grp)
            for i in range(n - seq_len - pred_len + 1):
                self._indices.append((station, i))

        if not self._indices:
            raise ValueError(
                f'No valid windows (seq_len={seq_len}, pred_len={pred_len}).'
            )

    def __len__(self):  return len(self._indices)

    def __getitem__(self, idx):
        station, i = self._indices[idx]
        x = self._feat[station][i : i + self.seq_len]
        y = self._tgt[station][i + self.seq_len : i + self.seq_len + self.pred_len]
        return x, y

    @property
    def n_features(self):  return next(iter(self._feat.values())).shape[1]

    @property
    def n_targets(self):   return next(iter(self._tgt.values())).shape[1]

    @property
    def n_stations(self):  return len(self._feat)


# ── Public API ────────────────────────────────────────────────────────────────

def build_dataloaders(
    data_dir: str,
    seq_len: int         = 72,
    pred_len: int        = 48,
    batch_size: int      = 64,
    num_workers: int     = 2,
    scaler_save_dir: str = None,
):
    """
    Full pipeline: folder of station CSVs → (train_loader, val_loader, test_loader).

    Args:
        data_dir        : path to the folder containing per-station CSV files
        seq_len         : look-back window in hours  (default 72)
        pred_len        : forecast horizon in hours  (default 48)
        batch_size      : DataLoader batch size      (default 64)
        num_workers     : DataLoader worker processes (default 2)
        scaler_save_dir : if set, saves all_scaler.pkl here

    Returns:
        train_loader, val_loader, test_loader, scaler, feat_cols
        Tensor shapes — x: (batch, seq_len, n_features)
                        y: (batch, pred_len, n_targets)
    """
    df, all_cols = _load_and_clean(data_dir)
    train_df, val_df, test_df = _split_by_time(df, TRAIN_FRAC, VAL_FRAC)
    train_df, val_df, test_df, scaler = _scale(train_df, val_df, test_df, all_cols)

    if scaler_save_dir:
        os.makedirs(scaler_save_dir, exist_ok=True)
        joblib.dump(scaler, os.path.join(scaler_save_dir, 'all_scaler.pkl'))

    ds_kw = dict(station_col=STATION_COL, feature_cols=all_cols,
                 target_cols=all_cols, seq_len=seq_len, pred_len=pred_len)
    dl_kw = dict(batch_size=batch_size, num_workers=num_workers, pin_memory=True)

    return (
        DataLoader(AQIDataset(train_df, **ds_kw), shuffle=True,  **dl_kw),
        DataLoader(AQIDataset(val_df,   **ds_kw), shuffle=False, **dl_kw),
        DataLoader(AQIDataset(test_df,  **ds_kw), shuffle=False, **dl_kw),
        scaler, all_cols,
    )

---
## 8 · Import Test

In [ ]:
import sys, os

DATASET_PY = '/content/drive/MyDrive/AQI_Project/dataset.py'

if not os.path.exists(DATASET_PY):
    raise FileNotFoundError(
        'dataset.py not found on Drive. Run the %%writefile cell in Section 7 first.'
    )

if '/content/drive/MyDrive/AQI_Project' not in sys.path:
    sys.path.insert(0, '/content/drive/MyDrive/AQI_Project')

if 'dataset' in sys.modules:
    del sys.modules['dataset']

from dataset import AQIDataset, build_dataloaders
print('Import OK.')

In [ ]:
tr, va, te, scaler, fc = build_dataloaders(DATA_DIR, scaler_save_dir=CKPT_DIR)

xb, yb = next(iter(tr))
print(f'train_loader  : {len(tr.dataset):,} windows, {len(tr)} batches')
print(f'val_loader    : {len(va.dataset):,} windows')
print(f'test_loader   : {len(te.dataset):,} windows')
print(f'x shape       : {xb.shape}   → (batch, 72, n_features)')
print(f'y shape       : {yb.shape}   → (batch, 48, n_targets)  ← all pollutants')
print(f'n_features    : {tr.dataset.n_features}')
print(f'n_targets     : {tr.dataset.n_targets}')
print(f'Pollutant cols: {fc}')
print('\ndataset.py is ready. Share the AQI_Project Drive folder with your teammates.')